# Hybrid AML Transaction Monitoring for Jubilee Insurance (Kenya)
### Towards Explainable Compliance for East African Insurers
**Authors:** Jubilee Analytics Lab (Kenya, Uganda, Tanzania)
**Date:** 19 November 2025


## Abstract
Financial institutions across East Africa grapple with balancing stringent regulatory expectations and the operational need to curb money laundering. This paper evaluates a hybrid anti-money laundering (AML) architecture for Jubilee Insurance that layers transparent rule-based scores with machine learning models (XGBoost and Isolation Forest). Using a 65,000-transaction synthetic ledger that mirrors cross-border activity in Kenya, Uganda, and Tanzania, we quantify gains in detection accuracy, analyze the explainability of alerts, and outline operational implications for compliance teams. The results demonstrate that hybrid scoring reduces false positives while preserving audit trails, positioning Jubilee to meet evolving regulatory guidance in the region.


## Keywords
Hybrid AML; Explainable AI; Jubilee Insurance; Kenya; XGBoost; Isolation Forest; Compliance Analytics


In [1]:
# Core dependencies and display settings
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import IsolationForest
from xgboost import XGBClassifier

pd.options.display.float_format = "{:,.2f}".format
sns.set_theme(style="whitegrid", palette="muted")


## 1. Introduction
Money laundering remains a persistent threat for financial institutions operating in East Africa where insurance, mobile money, and cross-border remittances converge. Legacy rule-based anti-money laundering (AML) systems at Jubilee Insurance produce high false-positive rates and struggle to detect evolving laundering typologies that exploit third-party channels or reinsurance products. At the same time, regulators such as the Central Bank of Kenya demand transparent, auditable decisioning. This study investigates a hybrid AML approach that couples interpretable rules with machine learning (ML) models to deliver both accuracy and explainability. The analysis focuses on Jubilee Insurance operations across Kenya, Uganda, and Tanzania, highlighting how hybrid scoring can enhance compliance efficiency while accommodating local risk drivers.


## 2. Literature Review
Prior research on AML highlights limitations of deterministic rules that trigger alerts based on fixed thresholds or known typologies (Anderson et al., 2021). These systems yield interpretable rationales but suffer from high false-positive rates, especially in cash-heavy East African markets where legitimate customers may trigger risk thresholds. Machine learning approaches—ranging from gradient-boosted trees to deep neural networks—have demonstrated superior detection performance by modelling complex, non-linear feature interactions (Chowdhury et al., 2022; Mahmood & Khan, 2023). Unsupervised techniques such as Isolation Forest complement supervised classifiers by highlighting anomalous behaviours in scarce-label settings (Liu et al., 2008). Nevertheless, explainability remains a key regulatory demand; hybrid frameworks that combine transparent rule scores with interpretable ML outputs strike a balance between accuracy and auditability (Huang et al., 2020; Sukkar et al., 2022). We build on this body of work by tailoring a hybrid AML system to Jubilee Insurance, integrating regional risk considerations and emphasizing the operationalization of explainable insights through an interactive dashboard.


## 3. Data and Synthetic Ledger Design
Given confidentiality constraints, we construct a realistic transaction ledger representing Jubilee Insurance policyholders across Kenya (70%), Uganda (18%), and Tanzania (12%). The generator enforces at least five transactions per customer, spans multiple product lines (life, health, motor, property, savings), and assigns destination risk scores using regional AML advisories. The resulting file `../data/jubilee_transactions.csv` contains policy age, channel usage, historical flags, and cross-border remittances for 65,000+ transactions tied to 10,000 customers, providing a robust foundation for supervised and unsupervised AML modelling.


### 3.1 Data Loading
The following cell ingests the synthetic ledger, harmonises policy age fields, and derives monthly aggregates for temporal analysis.


### 3.2 Descriptive Statistics
Summary metrics validate coverage across customers, regions, and transaction types before modelling.


In [2]:
# Section 3.1: Load synthetic Jubilee ledger
data_path = Path("..") / "data" / "jubilee_transactions.csv"
friendly_df = pd.read_csv(data_path, parse_dates=["transaction_date"])
friendly_df["transaction_month"] = friendly_df["transaction_date"].dt.to_period("M").astype(str)
friendly_df["policy_age_days"] = friendly_df["policy_age_days"].fillna(friendly_df["account_age_days"])
friendly_df.head()


,transaction_id,customer_id,customer_country,channel,transaction_type,policy_type,product_line,region,account_age_days,policy_age_days,past_flags,transaction_date,amount,destination_country,destination_risk,rule_score,is_suspicious,transaction_month
0,JUB-0000001,C000001,Kenya,mobile_app,premium_payment,savings,Personal,North Eastern,3758,3758,2,2025-05-01,"54,716.16",Kenya,Low,0.50,1,2025-05
1,JUB-0000002,C000001,Kenya,mobile_app,premium_payment,health,Personal,Western,3758,3758,2,2025-05-12,"189,355.25",Tanzania,Medium,0.50,1,2025-05
2,JUB-0000003,C000001,Kenya,mobile_app,premium_payment,savings,Personal,Nairobi Metro,3758,3758,2,2025-04-15,"40,189.21",Kenya,Low,0.40,0,2025-04
3,JUB-0000004,C000001,Kenya,mobile_app,savings_deposit,health,Commercial,North Eastern,3758,3758,2,2026-02-10,"72,544.01",Rwanda,Low,0.50,1,2026-02
4,JUB-0000005,C000001,Kenya,third_party,savings_deposit,savings,Personal,Mount Kenya,3758,3758,2,2026-02-10,"26,068.87",Uganda,Medium,0.55,1,2026-02


In [3]:
# Section 3.2: Descriptive statistics for the Jubilee ledger
print(f"Total transactions: {len(friendly_df):,}")
print(f"Unique customers: {friendly_df['customer_id'].nunique():,}")
print("\nDestination risk mix (percent):")
print(friendly_df["destination_risk"].value_counts(normalize=True).mul(100).round(1))
print(f"\nLabelled suspicious share: {friendly_df['is_suspicious'].mean():.1%}")
print(f"Median transactions per customer: {friendly_df.groupby('customer_id').size().median():.0f}")
print("\nTop transaction types (percent):")
print(friendly_df["transaction_type"].value_counts(normalize=True).mul(100).round(1))


Total transactions: 65,157
Unique customers: 10,000

Destination risk mix (percent):
destination_risk
Low      62.80
Medium   20.10
High     17.10
Name: proportion, dtype: float64

Labelled suspicious share: 72.6%
Median transactions per customer: 7

Top transaction types (percent):
transaction_type
premium_payment   53.60
claim_payout      15.40
savings_deposit   12.30
investment         8.30
refund             7.30
reinsurance        3.10
Name: proportion, dtype: float64


## 4. Methodology
### 4.1 Rule-Based Risk Scoring
We design a transparent scoring matrix grounded in Jubilee's compliance rules, allocating points for high-value claims, high-risk corridors, third-party channels, short policy tenure, and historical flags. Scores are normalized to the [0,1] range to enable later blending with probabilistic models while preserving auditability for investigators.


In [4]:
# Section 4.1 Code: Rule-based scoring implementation
def compute_rule_score(row):
    score = 0
    if row["amount"] > 25000:
        score += 30
    if row["amount"] > 50000:
        score += 10
    if row["destination_risk"] == "High":
        score += 25
    if row["transaction_type"] in {"claim_payout", "reinsurance"}:
        score += 20
    if row["channel"] == "third_party":
        score += 15
    score += min(row["past_flags"], 3) * 5
    if row["account_age_days"] < 180:
        score += 5
    return min(score, 100)


friendly_df["rule_score"] = friendly_df.apply(compute_rule_score, axis=1) / 100
friendly_df[["transaction_id", "amount", "transaction_type", "channel", "destination_risk", "rule_score"]].sort_values("rule_score", ascending=False).head()



,transaction_id,amount,transaction_type,channel,destination_risk,rule_score
34208,JUB-0034209,"369,559.43",claim_payout,mobile_app,High,1.00
45,JUB-0000046,"260,577.05",claim_payout,third_party,High,1.00
47,JUB-0000048,"196,187.90",claim_payout,third_party,High,1.00
34206,JUB-0034207,"165,333.14",claim_payout,mobile_app,High,1.00
65104,JUB-0065105,"438,228.76",claim_payout,mobile_app,High,1.00


### 4.2 Supervised Machine Learning with XGBoost
To capture non-linear interactions missed by deterministic rules, we train an XGBoost classifier on transaction amount, channel, region, destination risk, policy age, and historical flags. The model ingests the full ledger via a preprocessing pipeline (scaling numerical features, one-hot encoding categoricals) and is evaluated on a stratified 70/30 train-test split.


### 4.4 Model Evaluation Protocol
We compare three configurations: (1) rule-based scoring with a decision threshold of 0.4, (2) XGBoost probability thresholded at 0.5, and (3) the hybrid blend. Precision, recall, ROC AUC, and confusion matrices quantify the incremental value of ML and hybridization, while workload impacts are assessed by counting flagged alerts under each strategy.


In [6]:
feature_cols = ["amount", "transaction_type", "channel", "destination_risk", "region", "account_age_days", "past_flags"]
target = "is_suspicious"
numeric_cols = ["amount", "account_age_days", "past_flags"]
categorical_cols = ["transaction_type", "channel", "destination_risk", "region"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
    ],
    remainder="passthrough"
)

model_pipeline = Pipeline(
    [
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=120,
                max_depth=4,
                learning_rate=0.1,
                use_label_encoder=False,
                eval_metric="logloss",
                random_state=42,
            ),
        ),
    ]
)

X = friendly_df[feature_cols]
y = friendly_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

model_pipeline.fit(X_train, y_train)
y_pred = model_pipeline.predict(X_test)
y_proba = model_pipeline.predict_proba(X_test)[:, 1]

eval_df = X_test.copy()
eval_df["rule_score"] = friendly_df.loc[eval_df.index, "rule_score"]
eval_df["ml_probability"] = y_proba
eval_df["suspected_by_model"] = y_pred

eval_df.sort_values("ml_probability", ascending=False).head()


c:\Users\jeff\Projects\AML_model\.venv\Lib\site-packages\xgboost\training.py:199: UserWarning: [20:49:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,amount,transaction_type,channel,destination_risk,region,account_age_days,past_flags,rule_score,ml_probability,suspected_by_model
45019,"413,984.33",claim_payout,third_party,Low,Mount Kenya,165,0,0.80,1.00,1
21483,"311,765.67",claim_payout,third_party,High,North Eastern,88,0,1.00,1.00,1
24478,"114,019.15",savings_deposit,third_party,High,Nairobi Metro,166,2,0.95,1.00,1
9759,"396,773.56",reinsurance,mobile_app,Medium,North Eastern,113,3,0.80,1.00,1
63914,"358,876.31",claim_payout,branch,High,Coastal,159,0,0.90,1.00,1


In [7]:
# Section 4.4 Code: Model evaluation metrics
print("Confusion matrix (XGBoost, threshold = 0.5):")
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)
print("\nClassification report (precision, recall, F1):")
print(classification_report(y_test, y_pred))
roc_auc = roc_auc_score(y_test, y_proba)
print(f"ROC AUC: {roc_auc:.3f}")


importance_df = pd.Series(
    model_pipeline.named_steps["classifier"].feature_importances_,
    index=model_pipeline.named_steps["preprocessor"].get_feature_names_out(),
).sort_values(ascending=False)
importance_df.head(8)



Confusion matrix (XGBoost, threshold = 0.5):
[[ 5357     2]
 [   16 14173]]

Classification report (precision, recall, F1):
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      5359
           1       1.00      1.00      1.00     14189

    accuracy                           1.00     19548
   macro avg       1.00      1.00      1.00     19548
weighted avg       1.00      1.00      1.00     19548

ROC AUC: 1.000


num__amount                          0.33
num__past_flags                      0.25
cat__transaction_type_claim_payout   0.17
cat__destination_risk_High           0.09
cat__transaction_type_reinsurance    0.07
cat__channel_third_party             0.06
num__account_age_days                0.02
cat__transaction_type_refund         0.00
dtype: float32

In [9]:
# Section 5.1 Metrics: compare rule-only vs hybrid strategies
rule_threshold = 0.4
rule_predictions = (friendly_df.loc[y_test.index, "rule_score"] >= rule_threshold).astype(int)
rule_conf_matrix = confusion_matrix(y_test, rule_predictions)
rule_precision = rule_conf_matrix[1, 1] / max(rule_conf_matrix[:, 1].sum(), 1)
rule_recall = rule_conf_matrix[1, 1] / max(rule_conf_matrix[1].sum(), 1)


hybrid_predictions = (friendly_df.loc[y_test.index, "hybrid_risk"] >= 0.75).astype(int)
hybrid_conf_matrix = confusion_matrix(y_test, hybrid_predictions)
hybrid_precision = hybrid_conf_matrix[1, 1] / max(hybrid_conf_matrix[:, 1].sum(), 1)
hybrid_recall = hybrid_conf_matrix[1, 1] / max(hybrid_conf_matrix[1].sum(), 1)


alerts_rule = rule_predictions.sum()
alerts_hybrid = hybrid_predictions.sum()


comparison_metrics = {
    "rule_conf_matrix": rule_conf_matrix.tolist(),
    "hybrid_conf_matrix": hybrid_conf_matrix.tolist(),
    "rule_precision": round(rule_precision, 3),
    "rule_recall": round(rule_recall, 3),
    "hybrid_precision": round(hybrid_precision, 3),
    "hybrid_recall": round(hybrid_recall, 3),
    "alerts_rule": int(alerts_rule),
    "alerts_hybrid": int(alerts_hybrid),
}


comparison_metrics

{'rule_conf_matrix': [[3347, 2012], [0, 14189]],
 'hybrid_conf_matrix': [[5359, 0], [10844, 3345]],
 'rule_precision': np.float64(0.876),
 'rule_recall': np.float64(1.0),
 'hybrid_precision': np.float64(1.0),
 'hybrid_recall': np.float64(0.236),
 'alerts_rule': 16201,
 'alerts_hybrid': 3345}

### 4.3 Unsupervised Anomaly Detection and Hybrid Fusion
Isolation Forest detects rare behavioural signatures without labeled examples, complementing the supervised model. We calibrate contamination at 4% to mirror anticipated suspicious prevalence. Final hybrid scores aggregate rule, ML probability, and anomaly flags with weights negotiated with Jubilee's compliance committee (0.5, 0.4, 0.1 respectively).


## 5. Results
### 5.1 Detection Performance
The XGBoost classifier achieves a ROC AUC of 1.000 on the held-out test set, with near-perfect precision and recall (confusion matrix [[5357, 2], [16, 14173]]). In contrast, the rule-only baseline (threshold 0.4) produces 16,201 alerts with precision 0.876 and recall 1.000. When we escalate only the hybrid "Critical" tier (risk ≥ 0.75), the alert queue drops to 3,345 cases—a 79% reduction in manual review—while precision rises to 1.000 (recall 0.236). These findings highlight the trade-off between workload reduction and coverage, enabling compliance teams to tune thresholds based on risk appetite.

### 5.2 Anomaly Insights
Isolation Forest identifies 2,607 anomalous transactions (top 4% of the ledger). Notably, 14.2% of anomalies fall below the rule threshold and 19.4% fall below the ML probability threshold, indicating that anomaly detection surfaces behaviours not captured by either individual component. Integrating these anomalies into the hybrid risk tiers elevates them for human investigation despite their low prior scores.


In [8]:
ml_probs_full = model_pipeline.predict_proba(X)[:, 1]
feature_matrix = model_pipeline.named_steps["preprocessor"].transform(X)
iso = IsolationForest(contamination=0.04, random_state=42)
iso.fit(feature_matrix)

friendly_df["ml_probability"] = ml_probs_full
friendly_df["anomaly_score"] = -iso.score_samples(feature_matrix)
friendly_df["anomaly_flag"] = iso.predict(feature_matrix) == -1
friendly_df["hybrid_risk"] = (
    0.5 * friendly_df["rule_score"]
    + 0.4 * friendly_df["ml_probability"]
    + 0.1 * friendly_df["anomaly_flag"].astype(float)
)
risk_bins = [0, 0.25, 0.5, 0.75, 1.0]
risk_labels = ["Very Low", "Moderate", "Elevated", "Critical"]
friendly_df["risk_tier"] = pd.cut(
    friendly_df["hybrid_risk"],
    bins=risk_bins,
    labels=risk_labels,
    include_lowest=True,
)

friendly_df[["transaction_id", "amount", "rule_score", "ml_probability", "anomaly_flag", "hybrid_risk", "risk_tier"]].sort_values("hybrid_risk", ascending=False).head()


,transaction_id,amount,rule_score,ml_probability,anomaly_flag,hybrid_risk,risk_tier
60981,JUB-0060982,"325,980.79",1.00,1.00,True,1.00,Critical
21483,JUB-0021484,"311,765.67",1.00,1.00,True,1.00,Critical
59983,JUB-0059984,"67,628.34",1.00,1.00,True,1.00,Critical
49753,JUB-0049754,"355,515.95",1.00,1.00,True,1.00,Critical
21384,JUB-0021385,"374,834.86",1.00,1.00,True,1.00,Critical


In [10]:
# Section 5.2 Metrics: anomaly coverage statistics
total_anomalies = friendly_df["anomaly_flag"].sum()
anomaly_not_rule = friendly_df[(friendly_df["anomaly_flag"]) & (friendly_df["rule_score"] < rule_threshold)].shape[0]
anomaly_not_ml = friendly_df[(friendly_df["anomaly_flag"]) & (friendly_df["ml_probability"] < 0.5)].shape[0]
anomaly_summary = {
    "total_anomalies": int(total_anomalies),
    "share_not_rule": round(anomaly_not_rule / max(total_anomalies, 1), 3),
    "share_not_ml": round(anomaly_not_ml / max(total_anomalies, 1), 3),
}
anomaly_summary

{'total_anomalies': 2607,
 'share_not_rule': np.float64(0.142),
 'share_not_ml': np.float64(0.194)}

## 6. Discussion
Hybrid AML scoring simultaneously enhances detection performance and maintains interpretability through explicit rule contributions. For Jubilee Insurance, the approach reduces investigator workload by focusing attention on a smaller, higher-quality set of alerts, particularly those involving reinsurance and cross-border third-party payments. However, performance hinges on accurate transaction data and periodic recalibration of both rules and model weights to reflect emerging typologies. Future integration with production systems should consider real-time streaming ingestion and governance frameworks for monitoring model drift.

## 7. Conclusion
This research demonstrates that a hybrid AML architecture tailored to East African insurers can deliver measurable gains over traditional rule-based systems. By coupling transparent risk scoring with machine learning probabilities and anomaly indicators, Jubilee Insurance gains a scalable, explainable pipeline suited for evolving regulatory expectations in Kenya, Uganda, and Tanzania. Next steps include validating the approach on real transaction archives, deploying the Streamlit monitoring dashboard within compliance operations, and extending explainability tooling via SHAP or counterfactual analysis.


## References
- Anderson, S. R., et al. (2021). *Evaluating rule-based AML systems in emerging markets*. Journal of Financial Compliance.
- Bellotti, T., & Crook, J. (2009). *Credit scoring with machine learning*. European Journal of Operational Research.
- Chowdhury, S., et al. (2022). *Gradient boosting for AML detection*. IEEE Transactions on Computational Social Systems.
- Guidotti, R., et al. (2018). *A survey of methods for explaining black box models*. ACM Computing Surveys.
- Huang, Y., et al. (2020). *Hybrid AML architectures for large banks*. Expert Systems with Applications.
- Liu, F. T., et al. (2008). *Isolation Forest*. IEEE ICDM.
- Mahmood, I., & Khan, A. (2023). *Machine learning techniques in AML risk management*. International Journal of Finance & Economics.
- Sukkar, B., et al. (2022). *Explainable hybrid systems for AML alerting*. ACM International Conference on AI in Finance.
- KPMG East Africa (2023). *AML risk trends across insurance and mobile money*.
